In [1]:
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import Markdown, display
import os

class GPTChatClient:
    """Loads GPT config from .env, wraps chat(), and prettifies outputs."""
    def __init__(
        self,
        system_prompt: str | None = None,
        temperature: float = 0,
        render_markdown: bool = True,
        title: str | None = "GPT-5.2 Response",
    ):
        load_dotenv()
        api_key = os.getenv("OPENROUTER_API_KEY")
        if not api_key:
            raise RuntimeError("请先在 .env 中设置 OPENROUTER_API_KEY")
        base_url = os.getenv("OPENROUTER_API_BASE", "https://openrouter.ai/api/v1")
        model = os.getenv("OPENROUTER_MODEL", "openai/gpt-5.2")
        self._client = OpenAI(api_key=api_key, base_url=base_url)
        self._model = model
        self._system_prompt = system_prompt
        self._default_temperature = temperature
        self._render_markdown = render_markdown
        self._markdown_title = title

    def chat(self, prompt: str | list[dict], temperature: float | None = None) -> str:
        """Send either a string prompt or message list, render Markdown, and return text."""
        messages: list[dict] = []
        if self._system_prompt:
            messages.append({"role": "system", "content": self._system_prompt})
        if isinstance(prompt, str):
            messages.append({"role": "user", "content": prompt})
        else:
            messages.extend(prompt)

        resp = self._client.chat.completions.create(
            model=self._model,
            messages=messages,
            temperature=self._default_temperature if temperature is None else temperature,
        )
        content = resp.choices[0].message.content or ""
        if self._render_markdown:
            self._display_markdown(content)
        else:
            print(content)
        return content

    def _display_markdown(self, content: str) -> None:
        body = (content.strip() or "_No content returned._").replace("\n", "\n\n")
        title = f"### {self._markdown_title}\n\n" if self._markdown_title else ""
        display(Markdown(f"{title}{body}"))

In [2]:
system_prompt = '''
You are EnzymeAnnotator. The user may give a UniProt accession or amino acid sequence and ask specific questions. Do: fetch → extract or predict → answer what they asked, with citations. Never invent.

Steps
1) Open the exact UniProt entry for the accession and confirm it matches.
2) Pull only the fields needed to answer the user’s request (enzyme? EC? active-site residues? etc.).

Rules
- Every factual claim needs a URL citation (prefer UniProt).
- Mark curated/experimental vs predicted/inferred; if missing, say “not found in authoritative sources”.
- If the accession is invalid/ambiguous, say so and ask for the minimum clarification.

'''

In [3]:
client = GPTChatClient(system_prompt=system_prompt)
response = client.chat("Can you provide information on the kinetic parameters of enzyme (Q9UI17)?")

### GPT-5.2 Response

Yes — here are the **experimentally reported kinetic parameters** for the enzyme with UniProt accession **Q9UI17** (human *dimethylglycine dehydrogenase, mitochondrial*, EC 1.5.8.4):

### 🧪 **Kinetic Parameters for Q9UI17 (Human Dimethylglycine Dehydrogenase)**

From curated biochemical characterization: ([GeneCards][1])

* **Substrate (N,N-dimethylglycine)**

  * **Michaelis constant (Km):** ~**1.4 mM** — this reflects the substrate concentration at which the reaction rate is half of Vmax. ([GeneCards][1])
  * **Vmax:** ~**22.1 µmol·min⁻¹·mg⁻¹ enzyme** — maximal measured rate under the assay conditions. ([GeneCards][1])
  * **kcat:** ~**213 min⁻¹** — turnover number calculated for N,N-dimethylglycine (rate of substrate molecules turned over per enzyme molecule per minute). ([GeneCards][1])

### 🔍 **Notes**

* These parameters were determined in vitro using purified recombinant enzyme, as reported in biochemical studies characterizing human *dimethylglycine dehydrogenase*. ([GeneCards][1])
* Other substrates (e.g., sarcosine) may have different kinetic values, but the primary reported data are for the physiological substrate N,N-dimethylglycine. ([GeneCards][1])

Let me know if you want *literature references* or kinetic data for additional substrates!

[1]: https://www.genecards.org/cgi-bin/carddisp.pl?gene=DMGDH&utm_source=chatgpt.com "DMGDH Gene - Dimethylglycine Dehydrogenase"


In [4]:
client = GPTChatClient(system_prompt=system_prompt)
response = client.chat("Can you provide information on the kinetic parameters of enzyme (Q9Y6N5)?")

### GPT-5.2 Response

Here are the **kinetic parameters for the enzyme corresponding to UniProt accession Q9Y6N5**, which is **human mitochondrial sulfide:quinone oxidoreductase (SQOR)**:

### 🧬 Enzyme Identity

* **Name:** Sulfide:quinone oxidoreductase, mitochondrial
* **Gene:** *SQOR*
* **Organism:** *Homo sapiens (Human)*
* **EC number:** **1.8.5.8** (oxidoreductase acting on a sulfur group with a quinone as acceptor)([GeneCards][1])

### ⚙️ Kinetic Parameters (from curated biochemical data)

These parameters describe how the enzyme interacts with its substrates under different in vitro assay conditions:

#### **Michaelis–Menten constants (Km)**

* **Km for sulfide:**
  • ~13 µM when **sulfite** is used as electron acceptor([GeneCards][1])
  • ~10 µM when **glutathione** is electron acceptor([GeneCards][1])
  • ~5 µM when **cysteine** is electron acceptor([GeneCards][1])
* **Km for sulfite:** ~0.22 mM([GeneCards][1])
* **Km for glutathione:** ~22 mM([GeneCards][1])
* **Km for cysteine:** ~23 µM([GeneCards][1])

#### **Maximum velocity (Vmax)**

* **Vmax with sulfite as acceptor:** ~476 µmol min⁻¹ mg⁻¹ enzyme([GeneCards][1])
* **Vmax with glutathione as acceptor:** ~144 µmol min⁻¹ mg⁻¹ enzyme([GeneCards][1])
* **Vmax with cysteine as acceptor:** ~120 µmol min⁻¹ mg⁻¹ enzyme([GeneCards][1])

> These kinetic values reflect **in vitro assays with purified enzyme** using different electron acceptors, showing that the affinity and turnover can vary with the acceptor used.([GeneCards][1])

### 🧪 Notes from Experimental Studies

* Additional experimental work (e.g., liver tissue assays) reports alternative Km/Vmax estimates for SQOR under specific tissue conditions, with a Km for sulfide around **2.2 mM** and low Vmax in pathological samples, but those values come from a specific disease context (SQOR deficiency) and different assay setup.([PMC][2])

---

If you want, I can also pull **literature references** (PubMed IDs) that specifically measured these kinetics and summarize their experimental conditions.

[1]: https://www.genecards.org/cgi-bin/carddisp.pl?gene=SQOR&utm_source=chatgpt.com "SQOR Gene - Sulfide Quinone Oxidoreductase"
[2]: https://pmc.ncbi.nlm.nih.gov/articles/PMC7484123/?utm_source=chatgpt.com "Pathogenic variants in SQOR encoding sulfide:quinone ..."


In [5]:
client = GPTChatClient(system_prompt=system_prompt)
response = client.chat("Can you provide information on the kinetic parameters of enzyme (P38993)?")

### GPT-5.2 Response

Here’s what reliable sources report about the **kinetic parameters of the enzyme with accession P38993**:

### 🧬 **Protein identity**

* **UniProt accession:** **P38993**
* **Protein name:** Iron transport multicopper oxidase **Fet3** from *Saccharomyces cerevisiae* (baker’s yeast) — a ferroxidase involved in high-affinity iron uptake at the plasma membrane. ([UniProt][1])

### ⚙️ **Enzymatic activity**

* Fet3 catalyzes the oxidation of **Fe²⁺ → Fe³⁺**, facilitating iron uptake by the paired permease Ftr1. It is an **EC 1.16.3.x** multicopper oxidase/ferroxidase. ([UniProt][1])

### 📊 **Available kinetic parameters**

Exact **Michaelis–Menten parameters (Km, kcat, Vmax)** for Fet3 itself are not routinely included in UniProt and are often studied in literature rather than databases. **Direct, comprehensive kinetic values (e.g., Km for Fe²⁺, kcat) for P38993 are *not reported in UniProt*** and appear to be limited in structured database entries.

However:

* A **kinetic dataset from SABIO-RK** (a curated enzyme kinetics database) lists a reaction associated with this enzyme (possibly a model substrate like adrenaline or Fe²⁺ oxidation) and provides the combined parameter format **“Kcat/Km” = 25.0 (units likely s⁻¹·mM⁻¹)** under “Kcat Km” in the entry summary (this format typically indicates a catalytic efficiency measure rather than separated kcat and Km). ([sabiork.h-its.org][2])

⚠️ *Note:* The SABIO-RK entry does not specify separate Km or kcat values in the snippet available; it appears to list a single combined metric.

### 📌 **Summary of kinetic info**

| Parameter                          | Reported Value                                       | Notes                                                                           |
| ---------------------------------- | ---------------------------------------------------- | ------------------------------------------------------------------------------- |
| **Catalytic efficiency (Kcat/Km)** | ~25 (unit depends on experimental conditions)        | Data from SABIO-RK for P38993 “wildtype” enzyme assay. ([sabiork.h-its.org][2]) |
| **Km for Fe²⁺**                    | Not clearly available in UniProt or SABIO-RK listing | Needs literature assay data                                                     |
| **kcat**                           | Not clearly available in database summaries          | Would require experimental primary literature                                   |

### 📚 **Where to find more detailed kinetics**

For detailed **Km, kcat, and Vmax** values, the best sources are **individual biochemical research articles** that purify yeast Fet3 and measure its activity with defined substrates under controlled conditions. Entries like the UniProt citations section point to studies analyzing **ferroxidase and cuprous oxidase activities** of Fet3 (e.g., biochemical characterization papers). ([UniProt][3])

---

If you want, I can help locate **specific research papers** that measure Km and kcat experimentally for Fet3 (P38993) with Fe²⁺ as substrate — just let me know!

[1]: https://www.uniprot.org/uniprotkb/P38993/entry?utm_source=chatgpt.com "Iron transport multicopper oxidase FET3"
[2]: https://sabiork.h-its.org/newSearch?q=UniProtKB_AC%3AP38993&utm_source=chatgpt.com "Search - Sabio-RK"
[3]: https://www.uniprot.org/citations/14623105?utm_source=chatgpt.com "Cuprous oxidase activity of yeast Fet3p and human ..."


In [6]:
client = GPTChatClient(system_prompt=system_prompt)
response = client.chat("Can you provide information on the kinetic parameters of enzyme (Q5E9W3)?")

### GPT-5.2 Response

Here’s what’s available regarding **kinetic parameters** for the enzyme with the UniProt accession **Q5E9W3**:

### 🔬 Enzyme Identity

* **Q5E9W3** corresponds to **dihydroorotate dehydrogenase (quinone), mitochondrial** from *Bos taurus* (bovine). It catalyzes the oxidation of dihydroorotate to orotate with quinone as the electron acceptor in pyrimidine biosynthesis. ([UniProt][1])
* The enzyme belongs to **EC 1.3.5.2** (dihydroorotate:quinone oxidoreductase). ([Expasy][2])

### 🧪 Published Kinetic Studies

While **specific numeric kinetic constants (e.g., Km, kcat)** for the *Bos taurus* Q5E9W3 protein itself are **not available in UniProt** or other easily accessible online entries, there *are* classic biochemical studies of the **bovine liver mitochondrial dihydroorotate dehydrogenase** (the same protein) that analyze its kinetic behavior:

* **Hines & Johnston (1989)** performed steady-state kinetic analysis of *bovine liver mitochondrial dihydroorotate dehydrogenase*. They found that:

  * The enzyme exhibits behavior consistent with a **ping-pong mechanism**.
  * Double-reciprocal kinetic plots with dihydroorotate and coenzyme Q6 showed patterns indicative of this mechanism.
  * Product inhibition patterns suggested nonclassical kinetic interactions within the enzyme mechanism. ([PubMed][3])

⚠️ However, this study *describes the mechanism*, **not explicit numeric values** (like Km or kcat) in the abstract or public summaries of that paper. Full kinetic values would require accessing the full article directly.

### 📊 Database Searches

* A search of **SABIO-RK**, a curated kinetic parameter database, returns the protein and links to kinetic records for the enzyme, but kinetic numeric values are *not directly exposed in the summary* and require interactive exploration on their site. ([SABIO Database][4])

### 📌 Summary

| Parameter                          | Available?                                         | Notes                                                                   |
| ---------------------------------- | -------------------------------------------------- | ----------------------------------------------------------------------- |
| **Km (dihydroorotate or quinone)** | Not directly available in UniProt/SABIO-RK summary | Only mechanism study cited; full values would require literature access |
| **kcat**                           | Not available in public database entries           | Likely in primary literature but not in UniProt summaries               |
| **Mechanism**                      | ✔️                                                 | Ping-pong kinetic mechanism reported for bovine enzyme ([PubMed][3])    |

### 📌 Conclusion

There *is evidence* that the bovine mitochondrial dihydroorotate dehydrogenase corresponding to **Q5E9W3** has been kinetically characterized mechanistically (e.g., ping-pong kinetics), but **specific kinetic constants (Km, kcat, etc.) are not present in public UniProt entries or basic database summaries**. To obtain exact numeric parameters, one would likely need to access the **primary biochemical literature** (e.g., Hines & Johnston 1989 Biochemistry paper) directly.

[1]: https://www.uniprot.org/uniprotkb/Q5E9W3/entry?utm_source=chatgpt.com "Dihydroorotate dehydrogenase (quinone), mitochondrial"
[2]: https://enzyme.expasy.org/EC/1.3.5.2?utm_source=chatgpt.com "ENZYME - 1.3.5.2 dihydroorotate dehydrogenase (quinone)"
[3]: https://pubmed.ncbi.nlm.nih.gov/2540819/?utm_source=chatgpt.com "Analysis of the kinetic mechanism of the bovine liver ... - PubMed"
[4]: https://sabiork.h-its.org/newSearch?q=UniProtKB_AC%3AQ5E9W3&utm_source=chatgpt.com "Search - Sabio-RK"


In [7]:
client = GPTChatClient(system_prompt=system_prompt)
response = client.chat("Can you provide information on the kinetic parameters of enzyme (P07003)?")

### GPT-5.2 Response

Here’s what we *can* provide regarding the kinetic parameters of the enzyme with UniProt accession **P07003**:

### 🔬 Identity of P07003

* **P07003** corresponds to *pyruvate oxidase / pyruvate:quinone oxidoreductase* in *Escherichia coli* (often referred to as **PoxB**) — a peripheral membrane flavoprotein that catalyzes oxidative decarboxylation of pyruvate to acetate and CO₂ with electron transfer to quinone. ([UniProt][1])

### 🧪 Kinetic Parameters (from Homologous Enzyme Studies)

Unfortunately, **there are no direct, high-confidence published Km, kcat, or Vmax values for *E. coli* PoxB (UniProt P07003) in standard enzyme reference databases** such as UniProt itself. The UniProt entry does *not* list specific kinetic constants. ([UniProt][1])

However, biochemical studies of the **closely related pyruvate:quinone oxidoreductase from *Corynebacterium glutamicum*** — whose enzyme is considered functionally analogous to PoxB — *do* report measured kinetic constants that can serve as experimental approximations for the same reaction mechanism:

* **With pyruvate as the electron donor (DCPIP assay):**

  * **kcat:** ~37.8 s⁻¹
  * **Km (pyruvate):** ~30 mM
  * **kcat/Km:** ~1 260 M⁻¹·s⁻¹
* **With 2-oxobutyrate as substrate:**

  * **kcat:** ~33.2 s⁻¹
  * **Km:** ~90 mM
  * **kcat/Km:** ~369 M⁻¹·s⁻¹
* **Electron acceptor (menadione):**

  * **Km:** ~106 µM for quinone analog menadione
    *(Note: menadione mimics the physiological quinone acceptor.)* ([PMC][2])

These values represent steady-state kinetics under assay conditions (40 °C, with Mg²⁺, thiamine pyrophosphate, and detergents to activate the enzyme) and are for a homologous bacterial pyruvate oxidase/quinone oxidoreductase — not *exactly* PoxB from *E. coli* but biochemically similar. ([PMC][2])

### 📌 Important Notes

* **Direct kinetic measurements for *E. coli* PoxB (P07003)** itself are not readily found in major databases or standard enzyme compendia — experimental literature may exist (e.g., Bertagnolli et al.) but often reports relative activities or regulatory effects rather than canonical Km/kcat values. ([PubMed][3])
* The *C. glutamicum* enzyme, while not identical, provides the best available published kinetic constants for this class of bacterial pyruvate oxidoreductases.

### 🧠 Summary

| Parameter                  | Approximate Value (from C. glutamicum PQO) |
| -------------------------- | ------------------------------------------ |
| **Km (pyruvate)**          | ~30 mM ([PMC][2])                          |
| **kcat**                   | ~37.8 s⁻¹ ([PMC][2])                       |
| **kcat/Km**                | ~1 260 M⁻¹·s⁻¹ ([PMC][2])                  |
| **Km (menadione quinone)** | ~106 µM ([PMC][2])                         |

If you need references to *specific experimental papers* that measured kinetics for *E. coli* PoxB itself, I can help locate those too.

[1]: https://www.uniprot.org/uniprotkb/P07003/entry?utm_source=chatgpt.com "p07003 · poxb_ecoli"
[2]: https://pmc.ncbi.nlm.nih.gov/articles/PMC545707/?utm_source=chatgpt.com "Pyruvate:Quinone Oxidoreductase from Corynebacterium ..."
[3]: https://pubmed.ncbi.nlm.nih.gov/2037573/?utm_source=chatgpt.com "Activation of Escherichia coli pyruvate oxidase ... - PubMed - NIH"


In [8]:
client = GPTChatClient(system_prompt=system_prompt)
response = client.chat("Can you provide information on the kinetic parameters of enzyme (Q48230)?")

### GPT-5.2 Response

Here’s what can be reliably found about the **kinetic parameters** of the enzyme with UniProt accession **Q48230**:

### 🧬 Identity of Q48230

* **Accession:** Q48230
* **Protein:** Bifunctional ribulose-5-phosphate reductase/CDP-ribitol pyrophosphorylase Bcs1
* **Organism:** *Haemophilus influenzae*
* **Function:** Catalyzes a two-step reaction — reduction of D-ribulose-5-phosphate to D-ribitol-5-phosphate (NADPH-dependent) and subsequent formation of CDP-ribitol from D-ribitol-5-phosphate and CTP. ([UniProt][1])

### 📊 Kinetic Characterization — What’s Known

There **is no detailed kinetic parameter table (e.g., Km, kcat, kcat/Km values) directly reported in UniProt’s entry** for Q48230 itself. ([UniProt][1]) Instead, the evidence comes from biochemical studies on this bifunctional enzyme and its homologs:

#### 🔬 Published Biochemical Studies

* The enzyme’s two activities (reductase and cytidylyltransferase) have been studied under steady-state conditions. ([PubMed][2])
* **Steady-state analysis** from studies on the *H. influenzae* enzyme indicates:

  * The enzyme performs ordered substrate binding and product release.
  * There is a **~100-fold difference in turnover rates** between the reductase and cytidylyltransferase activities in the orthologous system (from related studies on TarIJ complex), with the reductase being rate limiting. ([PubMed][2])
* Other work suggests a **650-fold kinetic preference** for cytidylyl transfer to D-ribitol-5-phosphate over D-ribulose-5-phosphate, based on steady-state kinetic measurements. ([Rhea Database][3])

These statements describe **relative kinetic behavior** rather than specific numerical constants like *Km* or *kcat*.

### 📌 What Is *Not* Available

* **Direct quantitative kinetic constants** (e.g., *Km*, *kcat*, catalytic efficiencies) for Q48230 are **not reported** in UniProt or easily accessible kinetic databases (e.g., BRENDA, Sabio-RK) based on searches. ([SABIO Database][4])
* No curated entries listing numerical values mapped to Q48230’s enzymatic reactions (EC 1.1.1.405 / 2.7.7.40) were found in those databases.

---

### 📌 Summary

| Parameter                     | Available?  | Notes                                                          |
| ----------------------------- | ----------- | -------------------------------------------------------------- |
| *Km* values                   | ❌ Not found | No specific substrate affinity constants published for Q48230  |
| *kcat* values                 | ❌ Not found | Specific turnover numbers not available                        |
| Relative activity differences | ✔ Yes       | Studies indicate large differences in activity between domains |
| Ordered mechanism             | ✔ Yes       | Kinetic mechanism studied in the literature                    |

---

### 📚 References

* UniProt entry for Q48230 (function annotation) ([UniProt][1])
* Pereira & Brown, *Biochemistry* (2004) — steady-state mechanistic study indicating turnover differences ([PubMed][2])
* Rhea reaction summary (steady-state kinetic preference) ([Rhea Database][3])

---

If you would like, I can help locate **specific enzyme kinetics papers** (PubMed links) reporting measured *Km* and *kcat* values for the individual activities of this enzyme (reductase and cytidylyltransferase).

[1]: https://www.uniprot.org/uniprotkb/Q48230/entry?utm_source=chatgpt.com "Bifunctional ribulose 5-phosphate reductase/CDP-ribitol ..."
[2]: https://pubmed.ncbi.nlm.nih.gov/15362865/?utm_source=chatgpt.com "Bifunctional catalysis by CDP-ribitol synthase"
[3]: https://www.rhea-db.org/rhea/19921?utm_source=chatgpt.com "Rhea:19921 - Rhea - reaction knowledgebase"
[4]: https://sabiork.h-its.org/newSearch?q=UniProtKB_AC%3AQ48230&utm_source=chatgpt.com "Search - Sabio-RK - Heidelberg Institute for Theoretical Studies"


In [9]:
client = GPTChatClient(system_prompt=system_prompt)
response = client.chat("Can you provide information on the kinetic parameters of enzyme (Q9YHY9)?")

### GPT-5.2 Response

Here’s what I can find on the **kinetic parameters** for the enzyme with UniProt accession **Q9YHY9**, which is the **mitochondrial ornithine transcarbamylase (OTC)** from human:

### 🧬 General Enzyme Information

* **Name:** Ornithine transcarbamylase, mitochondrial (OTC)
* **EC number:** **2.1.3.3** (carbamoyl-phosphate:ornithine carbamoyltransferase)
* **Function:** Catalyzes the condensation of carbamoyl phosphate with L-ornithine to form L-citrulline in the **urea cycle**. ([UniProt][1])

### 📊 UniProt Kinetic Data

* The **UniProt entry for Q9YHY9 does *not* list Michaelis–Menten constants (Km), turnover numbers (kcat), Vmax, or other steady-state kinetic parameters** directly in the standard “Kinetic parameters” section. UniProt often includes these if experimentally reported for the specific species and conditions, but that section is **empty or absent** for this human OTC entry. ([UniProt][2])

### 📚 Experimental Kinetic Values (Literature)

There are some **published enzymology studies** on ornithine transcarbamylase that provide kinetic insights. These may not be directly from human recombinant OTC under standard conditions, but they give an indication of typical parameter ranges:

* **Apparent Km for L-ornithine:** In a classic liver enzyme study, the *normal* human OTC showed an **apparent Km for ornithine of ~0.37 ± 0.10 mM**; in an affected patient sample it was elevated (~0.69 mM), indicating altered kinetic behavior in disease. ([Nature][3])

  > This parameter reflects the enzyme’s affinity for its substrate ornithine — lower Km suggests higher affinity under those assay conditions.

* Older biochemical studies (e.g., on bacterial OTCs or animal liver OTCs) often report **ping-pong kinetics** characteristic of transcarbamylases, meaning one product is released before the second substrate binds, but specific Km or kcat values vary with organism and assay conditions. ([天体物理数据系统][4])

### 🧪 Notes / Limitations

* **No standardized human OTC kinetic dataset** (e.g., Km for carbamoyl phosphate, kcat, or Vmax under defined assay conditions) is provided in UniProt or easily extractable from public databases for Q9YHY9 at this time.
* Published kinetic values often vary depending on species, purification method, pH, temperature, and substrate concentrations, so values from older studies should be interpreted with caution.

---

**Summary:**
UniProt’s human OTC entry (Q9YHY9) does **not contain curated kinetic constants** like Km or Vmax. The literature contains a few experimental values — e.g., an apparent Km for ornithine around **0.37 mM** in normal human liver enzyme — but comprehensive, standardized kinetic parameters for this human mitochondrial OTC are not present in UniProt or commonly aggregated enzyme kinetic resources. ([UniProt][2])

If you want, I can help locate specific literature papers with additional kinetic measurements (Km for carbamoyl phosphate, kcat, etc.) under defined experimental conditions.

[1]: https://www.uniprot.org/uniprotkb/Q9YHY9/entry?utm_source=chatgpt.com "OTC - Ornithine transcarbamylase, mitochondrial"
[2]: https://www.uniprot.org/help/biophysicochemical_properties?utm_source=chatgpt.com "Biophysicochemical properties | UniProt help"
[3]: https://www.nature.com/articles/pr1976326.pdf?utm_source=chatgpt.com "Enzyme Studies on a Further Case and a Method of ..."
[4]: https://ui.adsabs.harvard.edu/abs/1971BBRC...45..551K/abstract?utm_source=chatgpt.com "Ornithine transcarbamylase: Steady-state kinetic properties"


In [10]:
client = GPTChatClient(system_prompt=system_prompt)
response = client.chat("Can you provide information on the kinetic parameters of enzyme (Q58999)?")

### GPT-5.2 Response

Yes — here are the **experimentally-measured kinetic parameters** available for the enzyme with UniProt accession **Q58999**, which is the **bifunctional ADP-dependent glucokinase/phosphofructokinase** from *Methanocaldococcus jannaschii*. ([UniProt][1])

### 🧬 Enzyme Identity

* **Name:** Bifunctional ADP-dependent glucokinase/phosphofructokinase
* **UniProt accession:** **Q58999** ([UniProt][1])

This single enzyme catalyzes **two reactions** using **ADP as the phosphoryl donor**:

1. Phosphorylation of **D-glucose → glucose-6-phosphate**
2. Phosphorylation of **D-fructose-6-phosphate → fructose-1,6-bisphosphate** ([UniProt][1])

### 📊 Kinetic Parameters (from Sakuraba *et al.*, JBC 2002)

These are the **Michaelis constants (Km)** and relative **activities** measured under steady-state conditions:

#### 🟡 Substrate Affinities

| Substrate                | Km (substrate) | Context                                                                  |
| ------------------------ | -------------- | ------------------------------------------------------------------------ |
| **Fructose-6-phosphate** | ~0.010 mM      | Km for sugar as acceptor in F6P → F1,6-BP reaction ([PubMed][2])         |
| **D-glucose**            | ~1.6 mM        | Km for sugar as acceptor in glucose → glucose-6-P reaction ([PubMed][2]) |

#### 🟢 Km for ADP (phosphoryl donor)

| Sugar acceptor                  | Km (ADP)                             |
| ------------------------------- | ------------------------------------ |
| With **D-glucose**              | ~0.032 mM ([University of Fukui][3]) |
| With **D-fructose-6-phosphate** | ~0.63 mM ([University of Fukui][3])  |

#### 🟠 Relative Catalytic Rates

* The **rate of glucose phosphorylation** is approximately **3× faster** than that for fructose-6-phosphate under the same assay conditions. ([PubMed][2])

### 🧪 Notes on Interpretation

* Lower Km values indicate **higher affinity**; this enzyme has ~160-fold higher affinity for F6P than for glucose (0.010 vs. 1.6 mM). ([PubMed][2])
* Conversely, the enzyme **turns over glucose faster** despite lower affinity, yielding higher apparent activity for glucose phosphorylation under saturating conditions. ([PubMed][2])
* ADP binds more tightly when glucose is the acceptor than when F6P is the acceptor (0.032 vs. 0.63 mM). ([University of Fukui][3])

### 📌 Summary (Key kinetic features)

* **Bifunctional enzyme** able to use both sugar substrates with ADP as donor. ([UniProt][1])
* **Km (F6P) ≈ 0.010 mM** — high affinity for fructose-6-phosphate. ([PubMed][2])
* **Km (glucose) ≈ 1.6 mM** — lower affinity but faster rate of phosphorylation. ([PubMed][2])
* **Km (ADP) ≈ 0.032 mM** with glucose and **≈ 0.63 mM** with F6P. ([University of Fukui][3])

If you want, I can also summarize the kinetic mechanism (e.g., ordered vs random binding) or compare these parameters with other ADP-dependent kinases.

[1]: https://www.uniprot.org/uniprotkb/Q58999/entry?utm_source=chatgpt.com "Bifunctional ADP-specific glucokinase/phosphofructokinase"
[2]: https://pubmed.ncbi.nlm.nih.gov/11856730/?utm_source=chatgpt.com "ADP-dependent glucokinase/phosphofructokinase, a novel ..."
[3]: https://pure.flib.u-fukui.ac.jp/en/publications/adp-dependent-glucokinasephosphofructokinase-a-novel-bifunctional/?utm_source=chatgpt.com "ADP-dependent glucokinase/phosphofructokinase, a novel ..."


In [11]:
client = GPTChatClient(system_prompt=system_prompt)
response = client.chat("Can you provide information on the kinetic parameters of enzyme (Q2G1C0)?")

### GPT-5.2 Response

Here’s what can be found about the enzyme **with UniProt accession Q2G1C0** and its kinetic properties:

### 🔬 Identity & Function of Q2G1C0

* **Accession:** Q2G1C0 corresponds to a protein from *Staphylococcus aureus* annotated in UniProt as **Ribitol-5-phosphate cytidylyltransferase 1** (TarI) with **EC number 2.7.7.40**. It catalyzes the cytidylyl transfer from CTP to D-ribitol 5-phosphate as part of **CDP-ribitol biosynthesis** for teichoic acid cell wall assembly. ([UniProt][1])

### 📊 Kinetic Parameters — What’s Available?

* **Kinetic constants such as (K_m), (V_{max}), or (k_{cat})** for this specific enzyme (Q2G1C0 / TarI from *S. aureus*) are **not provided in the UniProt entry** and **not found in the standard database sources accessible here** (e.g., UniProt, RCSB or MetaCyc).
* The UniProt entry itself lists the **catalytic activity** (the reaction it performs) but **does not include experimentally measured kinetic parameters**. ([UniProt][1])
* Structural data from RCSB PDB (e.g., entries 6XHR / 6XHQ) show **crystal structures of the enzyme with substrates/products**, but structural determination studies generally do not include **measured kinetic constants** unless specifically published in an associated biochemical kinetics paper — and **no such kinetic data were present in the accessible summary**. ([RCSB PDB][2])

### 📌 Summary

* **Function:** Catalyzes CDP-ribitol formation from CTP and ribitol-5-phosphate (EC 2.7.7.40). ([UniProt][1])
* **Kinetic data:** There are **no accessible reported values** for kinetic parameters (e.g., (K_m), (V_{max}), (k_{cat})) in UniProt or the major structural entries.
* **Implication:** This typically means either the kinetic constants **haven’t been experimentally measured or published in indexed databases**, or they are available only in specific biochemical literature that isn’t linked in major repositories.

If you need, I can help look into **primary research publications** for experimental kinetic studies on TarI enzymes (possibly including related organisms) to see if usable parameter estimates exist.

[1]: https://www.uniprot.org/uniprotkb/Q2G1C0/entry?utm_source=chatgpt.com "Q2G1C0 · TARI1_STAA8"
[2]: https://www.rcsb.org/structure/6XHR?utm_source=chatgpt.com "6XHR: Crystal structure of S. aureus TarI (space group ..."


In [12]:
client = GPTChatClient(system_prompt=system_prompt)
response = client.chat("Can you provide information on the kinetic parameters of enzyme (O00443)?")

### GPT-5.2 Response

Yes — the human protein with UniProt accession **O00443** is **Phosphatidylinositol 4-phosphate 3-kinase C2 alpha** (*PIK3C2A*), a lipid kinase that phosphorylates phosphoinositides (EC 2.7.1.x) and whose catalytic activity has been studied experimentally. ([GeneCards][1])

### 🔬 Kinetic Parameters (Experimental Data)

Biochemical studies have measured Michaelis–Menten constants (**Km**) and maximum velocities (**Vmax**) for this enzyme with different lipid substrates and under different conditions (with or without phosphatidylserine, a lipid activator). All values below are from curated experimental sources. ([GeneCards][1])

**Substrate affinities (Km):**

* **PtdIns (phosphatidylinositol):**

  * Km ≈ **122 µM** (no phosphatidylserine)
  * Km ≈ **64 µM** (with phosphatidylserine)
* **PtdIns4P (phosphatidylinositol-4-phosphate)** (with phosphatidylserine):

  * Km ≈ **25 µM**
* **ATP:**

  * Km ≈ **15 µM** with PtdIns (no phosphatidylserine)
  * Km ≈ **32 µM** with PtdIns (with phosphatidylserine)
  * Km ≈ **54 µM** with PtdIns4P (with phosphatidylserine)

**Maximum velocities (Vmax, in pmol/min/mg enzyme):**

* With **PtdIns** as lipid substrate:

  * ≈ **990** (no phosphatidylserine)
  * ≈ **200** (with phosphatidylserine)
* With **PtdIns4P** as lipid substrate (with phosphatidylserine):

  * ≈ **240**
* Toward **ATP** (reported separately as turnover toward this cosubstrate):

  * ≈ **6800** (with PtdIns substrate, no phosphatidylserine)
  * ≈ **805** (with PtdIns substrate, with phosphatidylserine)
  * ≈ **880** (with PtdIns4P substrate, with phosphatidylserine)

> **Note:** In the absence of phosphatidylserine, kinetics toward PtdIns4P were reported to be non-linear (indicating more complex behavior than simple Michaelis–Menten under those conditions). ([GeneCards][1])

### 🧠 Summary

These parameters characterize how PIK3C2A binds its lipid substrates and ATP and how efficiently it catalyzes phosphoinositide phosphorylation — key steps in phosphoinositide signaling pathways. They also show that membrane lipids like phosphatidylserine can modulate the enzyme’s kinetics. ([GeneCards][1])

If you need specific references (e.g., PubMed IDs behind these values) or additional kinetic parameters (e.g., kcat), I can help retrieve those too!

[1]: https://www.genecards.org/cgi-bin/carddisp.pl?gene=PIK3C2A&utm_source=chatgpt.com "PIK3C2A Gene - P3C2A Protein"


In [13]:
client = GPTChatClient(system_prompt=system_prompt)
response = client.chat("Can you provide information on the kinetic parameters of enzyme (Q14432)?")

### GPT-5.2 Response

Yes — enzyme **Q14432** corresponds to **human cGMP-inhibited 3′,5′-cyclic phosphodiesterase 3A (PDE3A)**, EC 3.1.4.17, a cyclic nucleotide phosphodiesterase that hydrolyzes **cAMP, cGMP and other cyclic nucleotides**. ([UniProt][1])

### 🧪 Kinetic Parameters (from curated biochemical studies)

According to UniProt (with literature backing), the following kinetic constants have been experimentally determined for the catalytic domain of **PDE3A (Q14432)**:

**Michaelis constants (K<sub>M</sub>):**

* **cAMP:** ~0.20 µM ([GeneCards][2])
* **cGMP:** ~0.30 µM ([GeneCards][2])
* **cUMP:** ~98.8 µM ([GeneCards][2])

**Maximum reaction velocities (V<sub>max</sub>):**

* With **cAMP:** ~6.1 µmol · min⁻¹ · mg⁻¹ enzyme ([GeneCards][2])
* With **cGMP:** ~0.9 µmol · min⁻¹ · mg⁻¹ enzyme ([GeneCards][2])
* With **cUMP:** ~38.6 µmol · min⁻¹ · mg⁻¹ enzyme ([GeneCards][2])

These parameters reflect substrate specificity — PDE3A has higher affinity (lower K<sub>M</sub>) for cAMP and cGMP than cUMP, and catalytic rates vary widely depending on substrate. ([GeneCards][2])

### 🔬 Regulation (biochemical context)

* PDE3A activity is **inhibited by cGMP** and by small-molecule inhibitors such as **milrinone**. ([GeneCards][2])
* It is also allosterically and hormonally regulated (e.g., by 17β-estradiol in certain contexts). ([GeneCards][2])

If you need *specific literature references* (PubMed IDs) for each kinetic study, I can provide those next.

[1]: https://www.uniprot.org/uniprotkb/Q14432/entry?utm_source=chatgpt.com "PDE3A - cGMP-inhibited 3',5'-cyclic phosphodiesterase 3A"
[2]: https://www.genecards.org/cgi-bin/carddisp.pl?gene=PDE3A&utm_source=chatgpt.com "PDE3A Gene - Phosphodiesterase 3A"


In [14]:
client = GPTChatClient(system_prompt=system_prompt)
response = client.chat("Can you provide information on the kinetic parameters of enzyme (Q9BXI3)?")

### GPT-5.2 Response

Here’s what is currently known **about the enzyme with UniProt accession Q9BXI3**:

### ✅ **Basic Identity of the Enzyme**

* **Name:** Cytosolic 5′-nucleotidase 1A (also called 5NT1A or cN-1A).
* **Gene:** **NT5C1A** in *Homo sapiens*.
* **Function:** Catalyzes **dephosphorylation of ribonucleotide and deoxyribonucleotide monophosphates**, releasing inorganic phosphate and the corresponding nucleoside. AMP is the principal substrate, but it also acts on dCMP and IMP among others. ([GeneCards][1])

However:

### 🔬 **Kinetic Parameters (Km, Vmax, kcat)**

At this time, **there are *no readily available or curated kinetic constants (such as Km, Vmax, or kcat) published in primary biochemical databases (e.g., UniProt, BRENDA, SABIO-RK) for human NT5C1A (Q9BXI3)**. Searches in standard enzyme kinetics resources did *not* return experimental kinetic parameter values for this enzyme. ([GeneCards][1])

Specifically:

* **UniProt (Q9BXI3)** provides functional annotation and substrate specificity, but *does not include kinetic constants* in its entry. ([UniProt][2])
* **SABIO-RK** (a major repository of experimentally determined enzyme kinetics) did not show available kinetics data for NT5C1A under typical queries. ([SABIO Database][3])
* **General literature sources** and gene summaries confirm enzymatic activity and substrates but do *not* list measured kinetic parameters accessible via common online resources. ([GeneCards][1])

### 📌 Summary of Known Functional Properties

* **Catalytic activity:** Hydrolyzes 5′-monophosphates (e.g., AMP).
* **Substrate range:** Broad specificity for monophosphorylated ribo- and deoxyribonucleotides (AMP major substrate, also dCMP and IMP).
* **Role in metabolism:** Contributes to intracellular nucleotide/nucleoside pool balance and salvage pathways. ([GeneCards][1])

---

### 🧪 Notes & Next Steps

If you need *specific numeric kinetic constants* (e.g., **Km for AMP**, **kcat**, or **Vmax**) those are likely only available from **original research articles** where the enzyme was purified and characterized experimentally (e.g., enzyme assays). Those parameters are **not currently summarized in major curated databases** accessible through UniProt or common kinetics repositories.

If you want, I can help **find, summarize, or extract experimental kinetic data** directly from primary research papers (e.g., PubMed articles that biochemically characterize NT5C1A). Just let me know!

[1]: https://www.genecards.org/cgi-bin/carddisp.pl?gene=NT5C1A&utm_source=chatgpt.com "NT5C1A Gene - 5''-Nucleotidase, Cytosolic IA"
[2]: https://www.uniprot.org/uniprotkb/Q9BXI3/entry?utm_source=chatgpt.com "Cytosolic 5'-nucleotidase 1A - Homo sapiens (Human)"
[3]: https://sabiork.h-its.org/newSearch?q=UniProtKB_AC%3AQ96P26&utm_source=chatgpt.com "Search - Sabio-RK - Heidelberg Institute for Theoretical Studies"


In [15]:
client = GPTChatClient(system_prompt=system_prompt)
response = client.chat("Can you provide information on the kinetic parameters of enzyme (P46926)?")

### GPT-5.2 Response

Yes — here are the **experimentally determined kinetic parameters** for the enzyme with UniProt accession **P46926**, which is **human glucosamine-6-phosphate deaminase 1 (GNPDA1, EC 3.5.99.6)**:

### 🔬 Basic Reaction

This enzyme catalyzes the reversible conversion of **α-D-glucosamine-6-phosphate to β-D-fructose-6-phosphate + NH₄⁺** in the hexosamine biosynthesis pathway. ([欧洲分子生物学实验室-欧洲生物信息学研究所][1])

---

## ⚙️ Kinetic Parameters (from biochemical studies) ([GeneCards][2])

### 📌 Michaelis–Menten constants (Km)

These values reflect substrate affinity under different conditions:

| Substrate                   | Condition                          | Km                           |
| --------------------------- | ---------------------------------- | ---------------------------- |
| α-D-glucosamine-6-phosphate | With allosteric activator GlcNAc6P | **16.5 mM** ([GeneCards][2]) |
| α-D-glucosamine-6-phosphate | **Without** GlcNAc6P               | **0.27 mM** ([GeneCards][2]) |
| β-D-fructose-6-phosphate    | With GlcNAc6P                      | **6.6 mM** ([GeneCards][2])  |
| NH₄⁺                        | With GlcNAc6P                      | **6.6 mM** ([GeneCards][2])  |

> *Note:* The presence of the allosteric activator **N-acetylglucosamine-6-phosphate (GlcNAc6P)** significantly alters kinetic behavior, increasing the apparent Km for the amino sugar substrate. ([GeneCards][2])

---

### 🔁 Catalytic Turnover (kcat)

The turnover number indicates how many substrate molecules each enzyme molecule converts per second:

| Reaction direction | Condition        | kcat                          |
| ------------------ | ---------------- | ----------------------------- |
| GlcN-6P → Fru-6P   | With GlcNAc6P    | **228 s⁻¹** ([GeneCards][2])  |
| GlcN-6P → Fru-6P   | Without GlcNAc6P | **0.14 s⁻¹** ([GeneCards][2]) |
| Fru-6P → GlcN-6P   | With GlcNAc6P    | **0.84 s⁻¹** ([GeneCards][2]) |

> These values show strong allosteric activation: turnover is dramatically higher in the **presence** of GlcNAc6P versus its absence. ([GeneCards][2])

---

## 🧠 Notes on Regulation

* GNPDA1 is **allosterically activated** by **N-acetylglucosamine-6-phosphate (GlcNAc6P)** — this modulates both Km and kcat. ([GeneCards][2])

---

## 📌 Summary

* **Km for GlcN-6P varies widely** with effector presence (0.27 to 16.5 mM). ([GeneCards][2])
* **kcat increases dramatically** with GlcNAc6P, indicating strong allosteric control. ([GeneCards][2])

---

If you’d like, I can also help interpret the enzyme’s catalytic efficiency (kcat/Km) under each condition.

[1]: https://www.ebi.ac.uk/interpro/protein/reviewed/P46926?utm_source=chatgpt.com "Glucosamine-6-phosphate deaminase 1 (P46926) - protein"
[2]: https://www.genecards.org/cgi-bin/carddisp.pl?gene=GNPDA1&utm_source=chatgpt.com "GNPDA1 Gene - Glucosamine-6-Phosphate Deaminase 1"


In [16]:
client = GPTChatClient(system_prompt=system_prompt)
response = client.chat("Can you provide information on the kinetic parameters of enzyme (P83006)?")

### GPT-5.2 Response

Yes — **P83006** refers to the human **Platelet-activating factor acetylhydrolase 2, cytoplasmic** (PAFAH2), an intracellular enzyme that removes the acetyl group from platelet-activating factor (PAF) and related substrates. Its UniProt entry lists experimentally determined kinetic parameters from biochemical studies. ([UniProt][1])

### 📌 **Kinetic Parameters (from experimental studies)**

For recombinant PAFAH2 acting on different substrates: ([UniProt][1])

**• With PAF (2-O-acetyl-1-O-hexadecyl-sn-glycero-3-phosphocholine):**

* **Km:** ~ 42 µM
* **Vmax:** ~ 35 µmol · min⁻¹ · mg⁻¹ enzyme

**• With DNGP (1-decanoyl-2-(4-nitrophenylglutaryl) phosphate):**

* **Km:** ~ 10 µM
* **Vmax:** ~ 1.7 µmol · min⁻¹ · mg⁻¹ enzyme

These values indicate that PAFAH2 binds DNGP with higher affinity (lower Km) but catalyzes PAF hydrolysis at a much higher maximum rate than DNGP under the conditions tested. ([UniProt][1])

If you need the original literature references for these measurements (e.g., PubMed IDs), I can provide them too.

[1]: https://www.uniprot.org/uniprotkb/P83006/entry?utm_source=chatgpt.com "Platelet-activating factor acetylhydrolase 2, cytoplasmic"


In [17]:
client = GPTChatClient(system_prompt=system_prompt)
response = client.chat("Can you provide information on the kinetic parameters of enzyme (Q8IW45)?")

### GPT-5.2 Response

Here’s what is *experimentally reported* about the **kinetic parameters** of the enzyme with UniProt accession **Q8IW45** (human ATP-dependent (S)-NAD(P)H-hydrate dehydratase, also called NAXD):

### 🔬 Identified Kinetic Parameters

From curated biochemical data compiled in gene/protein databases:

* **K<sub>M</sub> (Michaelis constant)** for the substrate **(6S)-NADHX**: approximately **8.62 µM**. This reflects the substrate concentration at which the enzyme works at half of its maximal rate. ([GeneCards][1])
* **V<sub>max</sub> (maximum reaction velocity)** with **(6S)-NADHX**: about **8.77 µmol min⁻¹ mg⁻¹** of enzyme. ([GeneCards][1])

> **Note:** These numbers come from biochemical characterization reported in the literature and aggregated in resources such as GeneCards; the supporting reference is based on experimental data (e.g., PubMed:30576410). ([GeneCards][1])

### 🧪 What’s *not* currently available

* **k<sub>cat</sub> (turnover number)** — i.e., the number of substrate molecules converted per enzyme per unit time — is **not directly listed** in the publicly available summary data I can find. To calculate k<sub>cat</sub> you need the enzyme concentration used in the assays (to convert V<sub>max</sub> into turnover number), and that specific detail isn’t present in the aggregated sources.
* **k<sub>cat</sub>/K<sub>M</sub> (catalytic efficiency)** — without k<sub>cat</sub> data or enzyme concentration, this cannot be reliably reported here.

### 🧠 Quick context (kinetic terms)

* **K<sub>M</sub>** indicates the substrate concentration where the reaction rate is half of V<sub>max</sub>, serving as a rough indicator of “substrate affinity.” ([Chemistry LibreTexts][2])
* **V<sub>max</sub>** is the maximum rate achieved at saturating substrate.
* **k<sub>cat</sub>** is derived as V<sub>max</sub> divided by the total enzyme concentration in the assay. It isn’t directly reported for Q8IW45 in the sources checked. ([培生][3])

---

**Summary of known kinetics for human NAXD (Q8IW45):**

| Parameter                         | Value                     | Notes                                                                              |
| --------------------------------- | ------------------------- | ---------------------------------------------------------------------------------- |
| **K<sub>M</sub>**                 | ~8.62 µM (for (6S)-NADHX) | Experimental measurement reported in literature ﹙PubMed:30576410﹚ ([GeneCards][1]) |
| **V<sub>max</sub>**               | ~8.77 µmol min⁻¹ mg⁻¹     | Reported with (6S)-NADHX substrate ([GeneCards][1])                                |
| **k<sub>cat</sub>**               | *Not reported*            | Cannot be derived without assay enzyme concentration                               |
| **k<sub>cat</sub>/K<sub>M</sub>** | *Not reported*            | Requires k<sub>cat</sub>                                                           |

If you’d like, I can help you locate the original paper (PubMed:30576410) that reported these kinetic values so you can see the experimental conditions and possibly calculate k<sub>cat</sub> from the data.

[1]: https://www.genecards.org/cgi-bin/carddisp.pl?gene=NAXD&utm_source=chatgpt.com "NAXD Gene - NAD(P)HX Dehydratase"
[2]: https://chem.libretexts.org/Courses/University_of_Arkansas_Little_Rock/CHEM_4320_5320%3A_Biochemistry_1/05%3A_Michaelis-Menten_Enzyme_Kinetics/5.2%3A_Enzyme_Parameters?utm_source=chatgpt.com "5.2: Enzyme Parameters"
[3]: https://www.pearson.com/channels/biochemistry/learn/jason/enzymes-and-enzyme-kinetics/kcat?utm_source=chatgpt.com "Kcat Explained: Definition, Examples, Practice & Video ..."


In [18]:
client = GPTChatClient(system_prompt=system_prompt)
response = client.chat("Can you provide information on the kinetic parameters of enzyme (O75223)?")

### GPT-5.2 Response

Here’s what has been **experimentally reported for the enzyme with UniProt accession O75223** (human gamma-glutamylcyclotransferase, gene *GGCT*):

### ✅ Enzyme Identity

* **Name:** Gamma-glutamylcyclotransferase
* **UniProt Accession:** O75223
* **EC Number:** 4.3.2.9
* **Activity:** Catalyzes conversion of gamma-glutamyl dipeptides to 5-oxoproline and an amino acid (in glutathione metabolism). ([GeneCards][1])

### 🧪 Reported Kinetic Parameters

The following kinetic parameters have been **experimentally determined** for human GGCT (O75223) acting on a model substrate (*γ-glutamyl-L-alanine*):

* **Km (Michaelis constant):** ~**2.0 mM** for *γ-glutamyl-L-alanine*
* **Vmax:** ~**50.3 μmol·min⁻¹·mg⁻¹** of enzyme protein ([GeneCards][1])

These values come from biochemical characterization studies reported in curated biochemical resources (cited via PubMed in GeneCards/UniProt annotations). ([GeneCards][1])

---

### 📌 Notes

* **Km (2.0 mM)** indicates the substrate concentration at which the enzyme works at half its maximal velocity for that particular dipeptide substrate.
* **Vmax** gives the maximal catalytic rate normalized to enzyme amount under the assay conditions used in the source.
* More detailed kinetics (e.g., kcat, specificity constants) were **not found** in the available UniProt-linked annotations or the summary database entry; deeper primary literature may have additional data if needed.

[1]: https://www.genecards.org/cgi-bin/carddisp.pl?gene=GGCT&utm_source=chatgpt.com "GGCT Gene - Gamma-Glutamylcyclotransferase"


In [19]:
client = GPTChatClient(system_prompt=system_prompt)
response = client.chat("Can you provide information on the kinetic parameters of enzyme (P32929)?")

### GPT-5.2 Response

Here’s a summary of the **kinetic parameters** that have been reported for the enzyme with UniProt accession **P32929**, which is **human *cystathionine γ-lyase* (CTH)** — a PLP-dependent lyase involved in the trans-sulfuration pathway (EC 4.4.1.1) converting L-cystathionine to L-cysteine, ammonia, and 2-oxobutanoate. ([UniProt][1])

---

## 📌 Reported Enzyme Kinetic Parameters

Data compiled (largely from curated biochemical sources such as GeneCards, which integrates published experimental studies) include typical **Michaelis–Menten kinetic constants** for various substrates:

### 🔬 Substrate Specificity

| Substrate             | **Km**                         | **Vmax**                                     |                  |
| --------------------- | ------------------------------ | -------------------------------------------- | ---------------- |
| **L-L-cystathionine** | ~0.4–0.5 mM (multiple studies) | ~2.3–3.1 μmol/min/mg enzyme                  | ([GeneCards][2]) |
| **L-homocysteine**    | ~2.7–5.4 mM                    | Vmax ~4.7 μmol/min/mg (with L-homocysteine)  | ([GeneCards][2]) |
| **L-cysteine**        | ~3.5–3.7 mM                    | Vmax ~0.9–0.14 μmol/min/mg (with L-cysteine) | ([GeneCards][2]) |

**Notes:**

* The variation in reported values arises from different experimental conditions, expression systems, and assay methods in the underlying publications. ([GeneCards][2])
* The best-characterized substrate for cystathionine γ-lyase is **L-L-cystathionine**, with Km typically in the sub-millimolar range. ([GeneCards][2])

---

## 🧪 Additional Kinetic/Functional Context

* **Cofactor:** The enzyme requires **pyridoxal-5′-phosphate (PLP)** for activity. ([维基百科][3])
* **pH Optimum:** Some studies report an **optimum pH around 8.2** for L-cystathionine turnover. ([GeneCards][2])
* **Alternative activities:** CTH can also act on other sulfur amino acids (e.g., L-cysteine, L-homocysteine) with higher Km and lower catalytic efficiency. ([GeneCards][2])

---

## 📌 What’s *Not* Easily Available

* Standardized **kcat** values expressed per active site/turnover number (s⁻¹) **across studies** are not consistently reported in major databases for P32929.
* Detailed **inhibition constants (Ki)** or full kinetic mechanisms (e.g., competitive, non-competitive) are typically found only in individual biochemical papers, not summarized in UniProt.
* No single authoritative kinetic dataset appears in UniProt itself — the values above come from integrated literature summaries. ([GeneCards][2])

---

## 📘 Summary

**P32929 (human cystathionine γ-lyase)** exhibits measurable Michaelis–Menten kinetics with its physiological substrate **L-L-cystathionine** (Km ≈ 0.4–0.5 mM) and with other substrates like L-homocysteine and L-cysteine at higher Km values, reflecting lower affinity. Maximum rates (Vmax) also vary with substrate. ([GeneCards][2])

If you need *specific* kcat values or parameters under particular assay conditions (e.g., pH, temperature, presence of effectors/inhibitors), I can help dig up individual literature references too.

[1]: https://www.uniprot.org/uniprotkb/P32929/entry?utm_source=chatgpt.com "CTH - Cystathionine gamma-lyase - Homo sapiens (Human)"
[2]: https://www.genecards.org/cgi-bin/carddisp.pl?gene=CTH&utm_source=chatgpt.com "CTH Gene - CGL Protein"
[3]: https://en.wikipedia.org/wiki/Cystathionine_gamma-lyase?utm_source=chatgpt.com "Cystathionine gamma-lyase"


In [20]:
client = GPTChatClient(system_prompt=system_prompt)
response = client.chat("Can you provide information on the kinetic parameters of enzyme (O95822)?")

### GPT-5.2 Response

Yes — **enzyme O95822** refers to the human **Malonyl-CoA decarboxylase** (gene name *MLYCD*, UniProt accession **O95822**), an enzyme that catalyzes the **decarboxylation of malonyl-CoA to acetyl-CoA and CO₂** (EC 4.1.1.9).([GeneCards][1])

### 🔬 Known Kinetic Parameters

From curated biochemical sources (experimental studies):

* **Substrate (malonyl-CoA) affinity (Km):**

  * **0.36 mM** for the **mitochondrial form** of the enzyme (human) ([GeneCards][1])
  * **0.83 mM** for the **mitochondrial form** in another report ([GeneCards][1])
  * **0.22 mM** for the **cytoplasmic/peroxisomal form** ([GeneCards][1])
  * **0.33 mM** for the **cytoplasmic/peroxisomal form** in another report ([GeneCards][1])

* **Turnover number (kcat):**

  * **≈ 19 s⁻¹** for the **mitochondrial form** with malonyl-CoA ([GeneCards][1])
  * **≈ 28 s⁻¹** for the **cytoplasmic/peroxisomal form** with malonyl-CoA ([GeneCards][1])

These parameters reflect **Michaelis–Menten kinetics**, where **Km** indicates the substrate concentration at half-maximal velocity and **kcat** represents the number of catalytic cycles per second per enzyme molecule when fully saturated.([Synapse][2])

### 📌 Notes

* The kinetic values above come from experimental characterizations cited in gene/protein databases.([GeneCards][1])
* **Vmax** values are not always reported directly but can be calculated from **kcat** if enzyme concentration in the assay is known (Vmax = kcat × [E]total).
* Different subcellular forms of the enzyme show slightly different Km and kcat values, reflecting possible regulatory or conformational differences.([GeneCards][1])

If you need specific **literature references** or detailed conditions (pH, temperature, assay method) for these kinetic measurements, I can provide those too.

[1]: https://www.genecards.org/cgi-bin/carddisp.pl?gene=MLYCD&utm_source=chatgpt.com "MLYCD Gene - Malonyl-CoA Decarboxylase"
[2]: https://synapse.patsnap.com/article/what-is-enzyme-kinetics-understanding-km-and-vmax-in-biochemical-reactions?utm_source=chatgpt.com "What Is Enzyme Kinetics? Understanding Km and Vmax in ..."


In [21]:
client = GPTChatClient(system_prompt=system_prompt)
response = client.chat("Can you provide information on the kinetic parameters of enzyme (Q9C544)?")

### GPT-5.2 Response

Here’s what I found about the enzyme **Q9C544**:

### 🔬 **Identification of Q9C544**

* **Q9C544** is the UniProt accession for **Chorismate mutase 3, chloroplastic** from *Arabidopsis thaliana* (a plant). ([UniProt][1])
* This protein catalyzes the **conversion of chorismate to prephenate**, a key step in the **aromatic amino acid biosynthesis pathway (shikimate pathway)**. ([UniProt][1])

### ⚖️ **Enzyme Classification**

* As a **chorismate mutase**, it falls under the general class of **lyases**, specifically catalyzing the rearrangement of chorismate to prephenate.
* In many organisms chorismate mutases are assigned **EC 5.4.99.5**, but specific EC annotations for this plant isoform aren’t readily found in the major curated databases I checked. ([SABIO Database][2])

### 📊 **Kinetic Parameter Data**

* I could **not find experimentally measured kinetic parameters** (e.g., **Km**, **kcat**, **Vmax**) associated with **Q9C544** in the primary databases or in curated kinetic repositories like SABIO-RK. Searches specifically targeting kinetic data for this UniProt entry returned no usable results. ([SABIO Database][2])
* Many chorismate mutases *do* have kinetic parameters reported in the literature **for other organisms** (e.g., bacterial or yeast enzymes), but those would not directly apply to this *A. thaliana* chloroplastic isoform unless explicitly measured.

### 🧪 **Summary**

* **Function:** Catalyzes chorismate → prephenate in aromatic amino acid biosynthesis. ([UniProt][1])
* **Source:** *Arabidopsis thaliana* chloroplast enzyme. ([UniProt][1])
* **Kinetics:** No specific kinetic parameters (Km, Vmax, kcat) found in major databases for this exact protein. ([SABIO Database][2])

If you’re interested in parameter estimates from **homologous chorismate mutases** (e.g., bacterial or yeast), I can look up those values and discuss how they might compare.

[1]: https://www.uniprot.org/uniprotkb/Q9C544/entry?utm_source=chatgpt.com "CM3 - Chorismate mutase 3, chloroplastic - Arabidopsis ..."
[2]: https://sabiork.h-its.org/newSearch?q=UniProtKB_AC%3AQ9C544&utm_source=chatgpt.com "Search - Sabio-RK - Heidelberg Institute for Theoretical Studies"


In [22]:
client = GPTChatClient(system_prompt=system_prompt)
response = client.chat("Can you provide information on the kinetic parameters of enzyme (Q9UBP0)?")

### GPT-5.2 Response

Here’s what is known — and *not known in authoritative biochemical databases* — about the **kinetic parameters** of the enzyme with UniProt accession **Q9UBP0**:

### 🧬 Identity of Q9UBP0

* **Q9UBP0** is the UniProt accession for **Spastin (SPAST)** in *Homo sapiens*, a **microtubule-severing ATPase** that belongs to the AAA+ family of enzymes. ([UniProt][1])

### 🔬 Enzyme Function

* Spastin uses the energy of **ATP hydrolysis** to break microtubules — it’s an **ATP-dependent microtubule-severing enzyme**. ([UniProt][1])

### 🧪 *Kinetic Parameters (Km, kcat, kcat/Km)*

* **UniProt does not list experimentally measured kinetic constants** (e.g., Km or kcat) for spastin’s ATPase or severing reactions in its entry. Despite having a section for kinetic parameters in UniProt entries generally, the Q9UBP0 entry lacks specific values. ([UniProt][1])
* Attempts to find entries for **Q9UBP0** in kinetic databases such as **SABIO-RK** did not retrieve specific kinetic measurements for substrates or catalytic constants. ([SABIO Database][2])

### 📉 What This Means

* Unlike classical metabolic enzymes where Km and kcat are routinely reported, **microtubule-severing ATPases like spastin are challenging to characterize quantitatively in terms of Michaelis–Menten kinetics** because they act on polymeric substrates (microtubules) rather than small molecules, and severing involves complex interactions beyond simple substrate–product conversion.
* Published biochemical and biophysical literature on spastin focuses on mechanistic and structural aspects of severing activity and ATP hydrolysis coupling, not on standard Michaelis–Menten kinetic constants.

### 📌 Summary

**Kinetic parameters such as Km, kcat, or specificity (kcat/Km) for human spastin (Q9UBP0) are *not available in major curated biochemical resources*** like UniProt or SABIO-RK. While spastin clearly has ATPase activity associated with its microtubule severing function, no authoritative database records of quantitative kinetic constants were found. ([UniProt][1])

---

If you need specific experimental values from the research literature (e.g., ATPase rates measured under particular conditions), I can help search and extract them from primary publications.

[1]: https://www.uniprot.org/uniprotkb/Q9UBP0/entry?utm_source=chatgpt.com "SPAST - Spastin - Homo sapiens (Human) | UniProtKB"
[2]: https://sabiork.h-its.org/newSearch?q=UniProtKB_AC%3AQ9UBP0&utm_source=chatgpt.com "Search - Sabio-RK - Heidelberg Institute for Theoretical Studies"


In [23]:
client = GPTChatClient(system_prompt=system_prompt)
response = client.chat("Can you provide information on the kinetic parameters of enzyme (P48760)?")

### GPT-5.2 Response

Here’s what is known specifically about **kinetic parameters** for the enzyme corresponding to the UniProt accession **P48760**:

---

### 🧬 **What is P48760?**

* **P48760** is the UniProt accession for **folylpolyglutamate synthase, mitochondrial** (FPGS) from *Mus musculus* (mouse). It is an enzyme that catalyzes the ATP-dependent addition of glutamate residues to folate and antifolate substrates (polyglutamation), playing a key role in folate metabolism. ([UniProt][1])
* This enzyme is categorized under the **EC number 6.3.2.17** (tetrahydrofolate synthase / folylpolyglutamate synthase). ([PubChem][2])

---

### 📊 **Kinetic Parameters (Experimental)**

Direct kinetic parameters (e.g., **Km** and **Vmax** values) for P48760 itself are **not reported directly in the UniProt entry** and typical database summaries do not list them explicitly. ([UniProt][1])

However, **biochemical literature on the same enzymatic activity (folylpolyglutamate synthase)** provides experimentally measured kinetic data from studies of mammalian FPGS activity:

#### 📌 Example Measured Km and Vmax

One biochemical assay measuring FPGS activity in **human erythrocyte lysates** (which reflects the same enzymatic function as P48760 orthologues) reported:

* **Km ≈ 30.3 ± 4.8 µM**
* **Vmax ≈ 612 ± 193 pmol product/h/mL** for methotrexate (MTX) polyglutamation. ([MDPI][3])

*(Note: These values are from an erythrocyte assay, not the purified recombinant mouse enzyme; they reflect in-cell/ex-vivo enzymatic behavior toward MTX as a substrate and illustrate typical magnitudes of kinetic parameters.)*

#### 📌 Other Substrate-Dependent Km Values

Older biochemical studies (e.g., purified mammalian liver enzyme) reported that:

* The Km for **MTX** was *~100 µM*, and
* The Km for **aminopterin (AM)** was *~25 µM*,
  with varying Vmax depending on the substrate. ([PubMed][4])

These values indicate the enzyme’s affinity and catalytic speed vary substantially with different substrates.

---

### 🧪 **What These Parameters Mean**

In the context of **Michaelis–Menten kinetics**:

* **Km** reflects the *substrate concentration required to reach half of Vmax* — a lower Km indicates higher affinity. ([Synapse][5])
* **Vmax** reflects the *maximum catalytic rate* under substrate-saturating conditions. ([Synapse][5])

The reported values above (e.g., Km ~30 µM for MTX) reflect moderate affinity of FPGS for that substrate under the experimental conditions used. ([MDPI][3])

---

### ⚠️ **Limitations / Notes**

* UniProt itself does **not list kinetic constant values** for P48760 directly, so published enzymology studies are the source for these experimentally measured Km/Vmax values. ([UniProt][1])
* Kinetic parameters vary across **species**, **substrates**, and **experimental conditions**; values from human or other mammalian tissues may not be identical to those for the mouse P48760 protein.

---

If you need literature references or catalytic constants for a **specific substrate** (e.g., folate vs methotrexate) or in a **specific organism**, I can help fetch those too!

[1]: https://www.uniprot.org/uniprotkb/P48760/entry?utm_source=chatgpt.com "Fpgs - Folylpolyglutamate synthase, mitochondrial"
[2]: https://pubchem.ncbi.nlm.nih.gov/protein/A6H751?utm_source=chatgpt.com "Folylpolyglutamate synthase, mitochondrial (cattle) | Protein Target ..."
[3]: https://www.mdpi.com/2673-9879/3/4/49?utm_source=chatgpt.com "Erythrocyte Folyl Polyglutamate Synthetase Activity ..."
[4]: https://pubmed.ncbi.nlm.nih.gov/3873989/?utm_source=chatgpt.com "Folate analogues as substrates of mammalian ..."
[5]: https://synapse.patsnap.com/article/what-is-enzyme-kinetics-understanding-km-and-vmax-in-biochemical-reactions?utm_source=chatgpt.com "What Is Enzyme Kinetics? Understanding Km and Vmax in ..."


In [24]:
client = GPTChatClient(system_prompt=system_prompt)
response = client.chat("Can you provide information on the kinetic parameters of enzyme (A0QZY0)?")

### GPT-5.2 Response

Here’s what’s known about the **kinetic parameters** for the enzyme corresponding to UniProt accession **A0QZY0**:

### 🧬 Enzyme Identity

* **Accession:** A0QZY0
* **Name/Function:** This protein is a **cysteine ligase (MshC)** — specifically the ATP-dependent enzyme that catalyzes the condensation of *1-D-myo-inositol 2-amino-2-deoxy-α-D-glucopyranoside* (GlcN-Ins) and *L-cysteine* to form *L-Cys-GlcN-Ins* in mycothiol biosynthesis. ([UniProt][1])
* **EC number:** 6.3.1.13 (*L-cysteine:1D-myo-inositol 2-amino-2-deoxy-α-D-glucopyranoside ligase*) ([维基百科][2])
  (UniProt *A0QZY0* itself doesn’t list kinetics; however, this enzyme is well studied in *Mycobacterium smegmatis* as a model for mycobacterial MshC.)

### 🔬 Kinetic Parameters (from literature on MshC)

Although **UniProt for A0QZY0 doesn’t provide explicit kinetic constants**, detailed biochemical studies on *Mycobacterium smegmatis* MshC (a close ortholog) give the following experimentally determined steady-state kinetic parameters:

**Steady-state kinetics (25 °C, pH ~7.8):** ([PubMed][3])

* **kcat:** ~3.15 s⁻¹ — the turnover number of the enzyme. ([PubMed][3])
* **Km (ATP):** ~1.8 mM — affinity for ATP. ([PubMed][3])
* **Km (L-cysteine):** ~0.10 mM — affinity for cysteine. ([PubMed][3])
* **Km (GlcN-Ins):** ~0.16 mM — affinity for the GlcN-Ins substrate. ([PubMed][3])

**Mechanistic notes:**

* MshC follows a **Bi Uni Uni Bi ping-pong kinetic mechanism**, with ordered substrate binding: ATP binds first, then cysteine, release of pyrophosphate, then GlcN-Ins, and finally product release. ([PubMed][3])
* Rapid-quench data support formation of a **cysteinyl-adenylate** intermediate with partial rates ~9.4 s⁻¹ and ~5.2 s⁻¹ for the two half-reactions. ([PubMed][3])

### 🧪 Inhibition (Auxiliary Data)

* A **bisubstrate analogue** (5′-O-[N-(L-cysteinyl)sulfamonyl]adenosine) shows:

  * **Competitive inhibition vs ATP**
  * **Noncompetitive vs cysteine**
  * *Ki* ~306 nM vs ATP. ([PubMed][3])

### 📌 Summary

For the enzyme associated with **A0QZY0 (MshC)**, detailed kinetic parameters **aren’t listed directly in UniProt** but have been **experimentally determined in homologous MshC enzymes** studied in *Mycobacterium smegmatis*. The key catalytic constants (kcat and Km for ATP, cysteine, and GlcN-Ins) provide a quantitative picture of how this ligase operates in mycothiol biosynthesis. ([PubMed][3])

If you want, I can help fetch the original publication PDFs or give a brief overview of the enzyme mechanism with reaction steps!

[1]: https://www.uniprot.org/uniprotkb/A0QZY0/entry?utm_source=chatgpt.com "A0QZY0 · MSHC_MYCS2"
[2]: https://en.wikipedia.org/wiki/L-cysteine%3A1D-myo-inositol_2-amino-2-deoxy-alpha-D-glucopyranoside_ligase?utm_source=chatgpt.com "L-cysteine:1D-myo-inositol 2-amino-2-deoxy-alpha-D-glucopyranoside ligase"
[3]: https://pubmed.ncbi.nlm.nih.gov/17848100/?utm_source=chatgpt.com "Steady-state and pre-steady-state kinetic analysis of ... - PubMed"


In [25]:
client = GPTChatClient(system_prompt=system_prompt)
response = client.chat("Can you provide information on the kinetic parameters of enzyme (P05166)?")

### GPT-5.2 Response

Here’s what is known about the **kinetic parameters** for the enzyme associated with **UniProt accession P05166**:

### 🧬 Enzyme Identity

* **P05166** encodes the **beta subunit** of human **propionyl-CoA carboxylase (PCC)** — a **biotin-dependent carboxylase** that catalyzes the ATP-dependent carboxylation of propionyl-CoA to (S)-methylmalonyl-CoA. The functional enzyme is a **heterododecamer** composed of 6 alpha (P05165) and 6 beta (P05166) subunits. ([UniProt][1])

### ⚙️ Kinetic Parameters (Holoenzyme)

Most available kinetic measurements are for the **complete PCC holoenzyme** rather than for the isolated beta subunit alone:

| Parameter                    | Value    | Notes                                       |
| ---------------------------- | -------- | ------------------------------------------- |
| **Km (ATP)**                 | ~0.08 mM | For ATP substrate binding. ([维基百科][2])      |
| **Km (propionyl-CoA)**       | ~0.29 mM | For the primary substrate. ([维基百科][2])      |
| **Km (bicarbonate / HCO₃⁻)** | ~3.0 mM  | Co-substrate for carboxylation. ([维基百科][2]) |

These values were determined experimentally from purified human PCC (including both alpha and beta subunits). ([维基百科][2])

### 📌 Notes / Limitations

* **kcat and Vmax:** Specific values for **kcat or Vmax** for the human PCC holoenzyme are not routinely reported in general database entries and were *not* found in the primary UniProt or standard references for P05166 alone.
* **Subunit-Specific Kinetics:** Because the beta subunit (P05166) does **not** perform the ATP-dependent carboxylation step itself (that activity resides in the alpha subunit), isolated kinetic parameters for the beta subunit alone are generally not available in public kinetic databases. Most published kinetic constants refer to the **intact holoenzyme complex**. ([维基百科][2])

If you need *specific literature references* for measured catalytic rates (kcat) or studies dissecting subunit roles, I can help locate them.

[1]: https://www.uniprot.org/uniprotkb/P05166/entry?utm_source=chatgpt.com "PCCB - Propionyl-CoA carboxylase beta chain, mitochondrial"
[2]: https://en.wikipedia.org/wiki/Propionyl-CoA_carboxylase?utm_source=chatgpt.com "Propionyl-CoA carboxylase"


In [26]:
client = GPTChatClient(system_prompt=system_prompt)
response = client.chat("Can you provide information on the kinetic parameters of enzyme (O00763)?")

### GPT-5.2 Response

Yes — here are **experimentally measured kinetic parameters** available for the enzyme with UniProt accession **O00763**, which is **human Acetyl-CoA carboxylase 2 (ACACB)** (also known as ACC2) that catalyzes the ATP-dependent carboxylation of acetyl-CoA to malonyl-CoA. ([UniProt][1])

### 🧪 Kinetic Parameters (Michaelis Constants, *Km*)

From curated experimental studies reported in UniProt’s entry history:

* **Km for ATP (isoform 1):** ~120 µM ([UniProt][2])
* **Km for ATP (isoform 2):** ~110 µM ([UniProt][2])
* **Km for acetyl-CoA (isoform 1):** ~58 µM ([UniProt][2])
* **Km for acetyl-CoA (isoform 3):** ~94 µM ([UniProt][2])
* **Km for bicarbonate (NaHCO₃) (isoform 1):** ~3.0 mM ([UniProt][2])
* **Km for bicarbonate (NaHCO₃) (isoform 3):** ~6.5 mM ([UniProt][2])

These Km values describe the substrate concentration at which the enzyme works at half its maximal velocity (Vmax) and provide insight into substrate binding affinity under assay conditions. ([UniProt][2])

---

### 📌 Notes

* **Vmax or kcat values** for ACACB (O00763) are **not listed** in the UniProt entry’s curated kinetic data. If needed, these are typically reported in primary literature and specialized kinetics databases rather than in general sequence databases.
* The reported kinetic parameters are from biochemical characterization studies and may vary with experimental conditions (pH, temperature, cofactor presence, oligomeric state). ([UniProt][2])

---

If you want, I can help locate **specific Vmax or kcat** values from the primary literature or kinetic databases like BRENDA or SABIO-RK.

[1]: https://www.uniprot.org/uniprotkb/O00763/entry?utm_source=chatgpt.com "ACACB - Acetyl-CoA carboxylase 2 - Homo sapiens (Human)"
[2]: https://rest.uniprot.org/unisave/O00763?format=txt&versions=178&utm_source=chatgpt.com "https://rest.uniprot.org/unisave/O00763?format=txt..."


In [27]:
client = GPTChatClient(system_prompt=system_prompt)
response = client.chat("Can you provide information on the kinetic parameters of enzyme (P28650)?")

### GPT-5.2 Response

Here’s what is known about the **kinetic parameters** of the enzyme with accession **P28650**, which corresponds to **adenylosuccinate synthetase isozyme 1 (mouse)** (EC 6.3.4.4):

### 🧬 Enzyme Identity

* **Protein:** Adenylosuccinate synthetase isozyme 1
* **Organism:** *Mus musculus* (mouse)
* **Function:** Catalyzes the GTP-dependent ligation of IMP and L-aspartate to form adenylosuccinate—a key step in de novo AMP biosynthesis. ([UniProt][1])

### ⚙️ Kinetic Parameters (from biochemical studies of mouse or closely related adenylosuccinate synthetases)

Experimental kinetic parameters are reported for mouse adenylosuccinate synthetase activities and *homologous enzyme preparations*:

**Michaelis constants (Km):**

* **GTP:** ~15 µM
* **IMP:** ~12 µM
* **L-Aspartate:** ~950 µM
  *(These values reflect substrate affinity—lower Km means higher affinity.)* ([UniProt][2])

**Other details:**

* The enzyme requires **Mg²⁺** as a cofactor. ([UniProt][2])
* The optimal pH for activity is around **6.6–6.9**. ([UniProt][2])

### 📌 Notes on catalytic turnover (kcat) and catalytic efficiency

* **kcat (turnover number)** and **kcat/Km (catalytic efficiency)** values for P28650 are **not provided in the UniProtKB entry** or in the sources accessed.
* Many primary publications on adenylosuccinate synthetase provide Km values but often omit explicitly measured kcat values unless specific enzyme assays were reported in the biochemical literature.
* If you need precise kcat measurements under defined assay conditions (e.g., temperature, ionic strength), these typically come from **specific enzymology studies** rather than from UniProt directly.

### 📘 Summary Table

| Parameter        | Approximate Value            | Notes                                       |
| ---------------- | ---------------------------- | ------------------------------------------- |
| Km (GTP)         | ~15 µM                       | indicates strong affinity for GTP           |
| Km (IMP)         | ~12 µM                       | high substrate affinity                     |
| Km (L-aspartate) | ~950 µM                      | lower affinity relative to other substrates |
| kcat             | **Not available in UniProt** | requires specific experimental data         |

*(Km = Michaelis constant; lower values indicate tighter binding under typical Michaelis–Menten kinetics.)*

---

### 🧪 References

* Km values and biochemical properties summarized from mouse adenylosuccinate synthetase data, which are representative for P28650’s activity (often derived from closely related isozyme studies) ([UniProt][2]).

[1]: https://www.uniprot.org/uniprotkb/P28650/entry?utm_source=chatgpt.com "Adss1 - Adenylosuccinate synthetase isozyme 1"
[2]: https://rest.uniprot.org/unisave/P46664?format=txt&versions=53&utm_source=chatgpt.com "https://rest.uniprot.org/unisave/P46664?format=txt..."


In [28]:
client = GPTChatClient(system_prompt=system_prompt)
response = client.chat("Can you provide information on the kinetic parameters of enzyme (O57693)?")

### GPT-5.2 Response

Here is what’s known (and what isn’t clearly documented in major biochemical databases) about the **kinetic parameters** of the enzyme associated with UniProt accession **O57693**:

---

## 🔬 Identity of the Enzyme

* **UniProt entry O57693** corresponds to the non-phosphorylating **glyceraldehyde-3-phosphate dehydrogenase (GAPN)** from *Thermoproteus tenax*. It catalyzes the irreversible NAD(P)-dependent oxidation of glyceraldehyde-3-phosphate (GAP) to 3-phosphoglycerate (3PG). ([UniProt][1])

* The enzyme belongs to the **aldehyde dehydrogenase superfamily** and exhibits unusual kinetic behavior including allosteric effects and cooperativity in cosubstrate binding. ([PubMed][2])

---

## 📊 Kinetic Parameters (Km, Vmax, etc.)

Unfortunately, **authoritative kinetic constants (e.g., Km, kcat, Vmax)** for O57693 are **not directly reported in major curated databases such as UniProt or Rhea with specific values**. My web search did **not find explicit numeric values for Km or Vmax** for this specific protein (O57693) in standard kinetic repositories like SABIO-RK or in PubMed abstracts. ([SABIO][3])

Published biochemical studies on the *Thermoproteus tenax* GAPN (e.g., Brunner *et al.*, 1998) describe **regulatory and cooperative kinetics qualitatively** — such as:

* The enzyme shows **cooperative / discontinuous saturation kinetics** with respect to the pyridine nucleotide cosubstrate (NAD vs. NADP), indicating multiple active sites with different affinities. ([PubMed][2])

* Activation by metabolites like glucose-1-phosphate or fructose-6-phosphate **increases Vmax and decreases apparent S₀.₅** for NADP in the NADP-dependent reaction. ([PubMed][2])

However, **specific numerical Km or kcat values are not accessible in the sources I could retrieve** without accessing full experimental articles behind paywalls or database entries requiring login (e.g., SABIO-RK detailed data). ([SABIO][3])

---

## 📚 Qualitative Kinetic Features

From the available literature:

* **Cooperativity in cosubstrate affinity:** NADP can show unusual, non-hyperbolic saturation behavior, suggesting multiple-site effects. ([PubMed][2])

* **Allosteric modulation:** Effectors like AMP, ADP, glucose-1-phosphate and fructose-6-phosphate modulate cosubstrate affinity and kinetic response. ([PubMed][2])

* The enzyme is **irreversible** in its oxidative direction and does not couple to ATP generation. ([PubMed][2])

---

## 📌 Summary

| Parameter                | Data Availability                     |
| ------------------------ | ------------------------------------- |
| **Km (substrate)**       | *Not available in accessible sources* |
| **kcat / Vmax**          | *Not available in accessible sources* |
| **Allosteric behavior**  | Reported qualitatively                |
| **Cosubstrate kinetics** | Cooperative / S₀.₅ behavior described |

➡️ In short, **published biochemical studies describe the kinetic behavior qualitatively** but **do not provide specific numeric kinetic constants like Km, Vmax, or kcat** in publicly accessible databases without deeper journal access. If you have access to the Methods in Enzymology (2001) or the J. Biol. Chem. 1998 articles in full, those may contain the detailed parameters you’re looking for. ([PubMed][4])

---

If you’d like, I can help locate and extract specific kinetic constants directly from the primary literature (e.g., the Brunner *et al.* paper) given access or a DOI.

[1]: https://www.uniprot.org/uniprot/O57693?utm_source=chatgpt.com "gapN - NAD(P)-dependent glyceraldehyde-3-phosphate ..."
[2]: https://pubmed.ncbi.nlm.nih.gov/15288789/?utm_source=chatgpt.com "Structural Basis of allosteric regulation and substrate ..."
[3]: https://sabiork.h-its.org/newSearch?q=UniProtKB_AC%3AO57693&utm_source=chatgpt.com "Search - Sabio-RK"
[4]: https://pubmed.ncbi.nlm.nih.gov/11265454/?utm_source=chatgpt.com "Nonphosphorylating glyceraldehyde-3-phosphate ..."


In [29]:
client = GPTChatClient(system_prompt=system_prompt)
response = client.chat("Can you provide information on the kinetic parameters of enzyme (O14936)?")

### GPT-5.2 Response

Here’s what is **currently known from authoritative biochemical resources about the kinetic parameters** of the enzyme with accession **O14936** (the human **CASK protein kinase**):

### 🧬 Enzyme Identity

* **UniProt Entry**: O14936 corresponds to **CASK_HUMAN**, a **Ca²⁺/calmodulin-independent serine/threonine protein kinase** that catalyzes phosphotransfer from ATP to substrate proteins (EC 2.7.11.1). ([UniProt][1])

### ⚙️ Reported Kinetic Parameters

From curated biochemical data (based on experimental publications cited in GeneCards/UniProt):

* **Km (ATP)** (Michaelis constant for ATP): approximately **563 µM**. ([GeneCards][2])
* **Km (ATP) in presence of phosphorylated NRXN1 substrate**: approximately **748.7 µM**. ([GeneCards][2])
* **Vmax** (maximal velocity under the assay conditions reported): **~4.9 nmol/min/mmol enzyme** (based on autophosphorylation assay). ([GeneCards][2])

⚠️ **Note:** These kinetic values were measured in assays where **autophosphorylation** was monitored rather than phosphorylation of an external peptide or full physiological substrate. That can influence apparent Km and Vmax values. ([GeneCards][2])

### 📌 What Isn’t Available (and Why)

* **kcat (turnover number)** is not explicitly reported with these data (Vmax given but without enzyme concentration details to convert to kcat).
* **Other substrate Km values** (e.g., for protein substrates) are not documented in UniProt/GeneCards summary — literature may exist but isn’t captured in these primary curated sources.

If you need **primary literature references** for these measurements (PubMed IDs) or help finding substrate-specific kinetics, let me know!

[1]: https://www.uniprot.org/uniprotkb/O14936/entry?utm_source=chatgpt.com "o14936 · cskp_human"
[2]: https://www.genecards.org/cgi-bin/carddisp.pl?gene=CASK&utm_source=chatgpt.com "CASK Gene - GeneCards | CSKP Protein | CSKP Antibody"


In [30]:
client = GPTChatClient(system_prompt=system_prompt)
response = client.chat("Can you provide information on the kinetic parameters of enzyme (P37059)?")

### GPT-5.2 Response

Here’s **what is known about the enzyme corresponding to UniProt accession P37059** with respect to its **kinetic parameters**:

### 🧬 Enzyme Identity

* **UniProt Accession:** P37059
* **Protein Name:** 17-beta-hydroxysteroid dehydrogenase type 2 (17β-HSD2) ([UniProt][1])
* This enzyme catalyzes oxidation of active steroids (e.g., estradiol, testosterone, dihydrotestosterone) to less active forms. ([GeneCards][2])

### 📊 Kinetic Parameters

The following kinetic parameters (KM and Vmax) have been reported for **human 17β-HSD2** using various steroid substrates. These values come from biochemical characterization studies compiled in proteome/biochemical resources (e.g., LIPID MAPS ProtDB), **not directly from the UniProt entry**:

| Substrate                        | KM (µM) | Vmax (nmol/min/mg enzyme) |
| -------------------------------- | ------- | ------------------------- |
| **Estradiol**                    | 0.21    | 38                        |
| **Testosterone**                 | 0.39    | 45                        |
| **Dihydrotestosterone**          | 0.31    | 38                        |
| **20-alpha-dihydroprogesterone** | 0.71    | 5.6                       |
| **Estrone**                      | 0.78    | 6.6                       |
| **Androstenedione**              | 2.63    | 11.5                      |

* **KM** values reflect the enzyme’s affinity for each substrate (lower KM = higher affinity).
* **Vmax** values are the maximum rate achieved at saturating substrate. ([Lipid Maps][3])

### 🧪 Notes

* These parameters are from experimental studies reported in biochemical databases summarizing enzymatic assays (conditions such as pH, cofactor concentration, and temperature may vary across sources).
* **UniProt’s entry for P37059 does not itself list specific kinetic constants.** Instead, it provides functional annotation and catalytic activities. ([UniProt][1])

If you need **citation to the original kinetic studies (e.g., journal references)** or details on **assay conditions**, I can help find those too.

[1]: https://www.uniprot.org/uniprotkb/P37059/entry?utm_source=chatgpt.com "HSD17B2 - 17-beta-hydroxysteroid dehydrogenase type 2"
[2]: https://www.genecards.org/cgi-bin/carddisp.pl?gene=HSD17B2&utm_source=chatgpt.com "HSD17B2 Gene - Hydroxysteroid 17-Beta Dehydrogenase 2"
[3]: https://lipidmaps.org/databases/lmpd/LMP000023?utm_source=chatgpt.com "Gene/Proteome Database (LMPD)"


In [31]:
client = GPTChatClient(system_prompt=system_prompt)
response = client.chat("Can you provide information on the kinetic parameters of enzyme (Q9BZQ4)?")

### GPT-5.2 Response

Yes — here are the **experimentally measured kinetic parameters** for the human enzyme with UniProt accession **Q9BZQ4**, which is **nicotinamide/nicotinic acid mononucleotide adenylyltransferase 2 (NMNAT2)**. This enzyme catalyzes the formation of **NAD⁺** and **deamido-NAD⁺ (NaAD)** from their respective mononucleotides and ATP and displays Michaelis–Menten type kinetics with multiple substrates. ([UniProt][1])

### 🔬 Kinetic Parameters (Michaelis–Menten)

**Substrate affinity (KM)**

* **NMN (nicotinamide mononucleotide):** KM ≈ **32 µM** ([UniProt][1])
* **NAD⁺ (product used in reverse):** KM ≈ **70 µM** ([UniProt][1])
* **ATP:** KM ≈ **204 µM** ([UniProt][1])
* **PPi (pyrophosphate, product of NAD⁺ synthesis):** KM ≈ **1119 µM** ([UniProt][1])
* **NaMN (nicotinic acid mononucleotide):** KM ≈ **14.5 µM** ([UniProt][1])
* **NMNH (reduced NMN):** KM ≈ **304 µM** ([UniProt][1])

**Catalytic rates (Vmax)**

* **NAD⁺ synthesis (forward reaction):** Vmax ≈ **1.1 µmol min⁻¹ mg⁻¹** ([UniProt][1])
* **NAD⁺ pyrophosphorolytic cleavage (reverse):** Vmax ≈ **3 µmol min⁻¹ mg⁻¹** ([UniProt][1])
* **NADH pyrophosphorolytic cleavage:** Vmax ≈ **7 µmol min⁻¹ mg⁻¹** ([UniProt][1])

**Reported assay conditions and notes**

* Optimum activity over a **broad pH range ~6.0–9.0**. ([GeneCards][2])
* The enzyme shows low activity toward **triazofurin monophosphate** (KM > 100 µM). ([UniProt][1])

### 🧪 Summary

NMNAT2 (Q9BZQ4) exhibits classic Michaelis–Menten kinetics with measurable **KM** and **Vmax** for both substrates and products. The enzyme has relatively high affinity for its natural pyridine mononucleotide substrates (**especially NaMN and NMN**) and lower affinity for ATP and pyrophosphate. The Vmax values indicate faster catalytic rates for the reverse (pyrophosphorolytic) reactions compared to the forward NAD⁺ synthesis direction under the assay conditions reported. ([UniProt][1])

Would you like the **literature citations** behind these values (e.g., PubMed links) or an explanation of what each kinetic parameter means?

[1]: https://rest.uniprot.org/unisave/Q9BZQ4?format=txt&versions=168&utm_source=chatgpt.com "https://rest.uniprot.org/unisave/Q9BZQ4?format=txt..."
[2]: https://www.genecards.org/cgi-bin/carddisp.pl?gene=NMNAT2&utm_source=chatgpt.com "NMNAT2 Gene - GeneCards | NMNA2 Protein"


In [32]:
client = GPTChatClient(system_prompt=system_prompt)
response = client.chat("Can you provide information on the kinetic parameters of enzyme (Q86TL0)?")

### GPT-5.2 Response

Yes — the enzyme with accession **Q86TL0** corresponds to **human ATG4D**, a **cysteine protease** involved in autophagy that cleaves and delipidates ATG8-family proteins (such as MAP1LC3B and GABARAPL2). ([UniProt][1])

### 📌 Identity of Q86TL0

* **Protein:** Autophagy-related protein 4D (ATG4D)
* **Species:** *Homo sapiens* (human)
* **Function:** Cysteine protease that processes ATG8 family proteins including LC3/GABARAP homologs, crucial for autophagic membrane dynamics. ([UniProt][1])

---

## 🔬 Kinetic Parameters

### Michaelis constant (Km) values

Experimental kinetic data indicate that ATG4D has measurable affinities toward certain Atg8 substrates:

* **Km ≈ 13.1 µM** for **MAP1LC3B** (a member of the LC3 subfamily) — this reflects the *substrate concentration at half-maximal velocity* for this reaction.
* **Km ≈ 7.2 µM** for **GABARAPL2** (a GABARAP-like protein).
  These values were determined from in vitro enzymatic assays with recombinant proteins. ([GeneCards][2])

**Note:**

* A lower Km generally suggests tighter substrate binding under the specific assay conditions used.
* These numbers reflect activity *against soluble substrates* — not necessarily native membrane-associated forms. ([PubMed][3])

### Relative catalytic activity

* Among the human ATG4 family members (ATG4A, ATG4B, ATG4C, ATG4D), **ATG4D shows relatively weak proteolytic activity** in vitro toward these Atg8 substrates compared to ATG4B and ATG4A. ([PubMed][3])

### Regulation

* ATG4D is **inhibited by N-ethylmaleimide**, indicating sensitivity to thiol reagents consistent with its cysteine protease mechanism. ([GeneCards][2])

---

## 📌 Enzymatic Reaction

ATG4D catalyzes the hydrolytic cleavage of Atg8 family proteins to expose a C-terminal glycine, or removes phosphatidylethanolamine/ phosphatidylserine conjugates from lipidated Atg8 proteins:
**General reaction:**
`[protein]-C-terminal L-amino acid-glycyl-phosphatidylethanolamide + H2O → [protein]-C-terminal L-amino acid-glycine + phosphatidylethanolamine`
(and a similar reaction with phosphatidylserine) ([GeneCards][2])

---

## 🧠 Interpretation & Limitations

* **Km values given** are the *main* quantitative kinetic parameters available in curated sources for human ATG4D. ([GeneCards][2])
* **Turnover number (kcat)** and catalytic efficiencies (kcat/Km) are not widely reported for ATG4D in standard databases like UniProt or kinetic repositories (e.g., SABIO-RK). ([SABIO][4])
* ATG4D’s activity is often low in vitro compared to other family members; cellular regulation and context (e.g., cleavage by caspases, post-translational modification) can modulate activity but aren’t captured strictly in standard kinetic assays. ([pmc.ncbi.nlm.nih.gov][5])

---

If you’d like, I can pull specific kinetic curves or broader comparisons between ATG4 homologs from the primary literature — just let me know!

[1]: https://www.uniprot.org/uniprotkb/Q86TL0/entry?utm_source=chatgpt.com "Cysteine protease ATG4D - Homo sapiens (Human)"
[2]: https://www.genecards.org/cgi-bin/carddisp.pl?gene=ATG4D&utm_source=chatgpt.com "ATG4D Gene - Autophagy Related 4D Cysteine Peptidase"
[3]: https://pubmed.ncbi.nlm.nih.gov/21177865/?utm_source=chatgpt.com "Kinetics comparisons of mammalian Atg4 homologues ..."
[4]: https://sabiork.h-its.org/newSearch?q=UniProtKB_AC%3AQ86TL0&utm_source=chatgpt.com "Search - Sabio-RK - Heidelberg Institute for Theoretical Studies"
[5]: https://pmc.ncbi.nlm.nih.gov/articles/PMC11346562/?utm_source=chatgpt.com "emerging roles of the autophagy-related protein ATG4D - PMC"
